In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, StableDiffusionXLPipeline, AutoencoderKL
from diffusers.utils import load_image
import numpy as  np
from PIL import Image, ImageOps

In [ ]:
controlnet_conditioning_scale = 0.9 

In [ ]:
controlnet = ControlNetModel.from_pretrained(
    "ShermanG/ControlNet-Standard-Lineart-for-SDXL",
    torch_dtype = torch.float16
)

In [ ]:
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix", 
    torch_dtype=torch.float16
)

In [ ]:
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet = controlnet,
    vae=vae,
    torch_dtype = torch.float16
).to("cuda")

In [ ]:
pipe.enable_model_cpu_offload()

In [ ]:
lineart = Image.open("/content/drive/MyDrive/深層学習/difusser/lineart.png").convert("RGB")

In [ ]:
# 白背景黒線なら反転（重要）
lineart = ImageOps.invert(lineart)

In [ ]:
lineart = lineart.resize((1024, 1024))

In [ ]:
lineart

In [ ]:


# --- 生成 ---
image = pipe(
    prompt="anime style coloring, vibrant colors, clean shading",
    negative_prompt="blurry, low quality, messy",
    image=lineart,
    controlnet_conditioning_scale=1.0,
    num_inference_steps=30,
    width=512,
    height=512
).images[0]

image.save("/content/output.png")